<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [2]</a>'.</span>

In [111]:
import pandas as pd
from datetime import datetime, timezone


import papermill as pm

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [112]:
pm.execute_notebook('data_consolidation.ipynb', 'data_consolidation.ipynb')

Executing:   0%|          | 0/23 [00:00<?, ?cell/s]

{'cells': [{'cell_type': 'code',
   'execution_count': 1,
   'id': '908b51fe',
   'metadata': {'execution': {'iopub.status.busy': '2026-03-12T23:41:31.372850Z',
     'iopub.execute_input': '2026-03-12T23:41:31.373052Z',
     'iopub.status.idle': '2026-03-12T23:41:32.534128Z',
     'shell.execute_reply': '2026-03-12T23:41:32.533849Z'},
    'papermill': {'exception': False,
     'start_time': '2026-03-12T23:41:31.367861+00:00',
     'end_time': '2026-03-12T23:41:32.536076+00:00',
     'duration': 1.168215,
     'status': 'completed'},
    'tags': []},
   'outputs': [],
   'source': "# Imports\nimport pandas as pd\nimport papermill as pm #papermill allows us to execute notebooks not just .py files.\nimport sys\nimport os\nsys.path.append(os.path.abspath('..')) \nfrom data_collection import fred_data_download as fred_data_download"},
  {'cell_type': 'code',
   'execution_count': 2,
   'id': '79aba66c',
   'metadata': {'execution': {'iopub.status.busy': '2026-03-12T23:41:32.547957Z',
     '

In [113]:
### Execute api pulls
pm.execute_notebook('pull_api.ipynb', 'pull_api_output.ipynb')

### Read in data

#load in kalshi ticker data
kalshi_df = pd.read_csv("../data/kalshi_full_data.csv")

#load in kalshi markets data to get open and close times for filtering
markets_df = pd.read_csv("../data/kalshi_markets.csv")

#select columns with open and close times
markets_df_cols = markets_df[["close_time", "open_time", "ticker"]]

Executing:   0%|          | 0/3 [00:00<?, ?cell/s]

In [114]:
### Manipulate datetime to get closest ticker to timestamp (eliminate bets for future fed decisions)

# extract ticker suffix to indicate which bet is being made
kalshi_df['ticker_suffix'] = kalshi_df['ticker'].str.split('-').str[-1]

# merge to get open and close times for each ticker
kalshi_w_markets = pd.merge(kalshi_df, markets_df_cols, left_on='ticker', right_on='ticker', how='left')

# parse as UTC then convert to Eastern Time
kalshi_w_markets['close_time'] = pd.to_datetime(kalshi_w_markets['close_time'], utc=True).dt.tz_convert('US/Eastern')
kalshi_w_markets['open_time'] = pd.to_datetime(kalshi_w_markets['open_time'], utc=True).dt.tz_convert('US/Eastern')
kalshi_w_markets['timestamp'] = pd.to_datetime(kalshi_w_markets['timestamp'], utc=True).dt.tz_convert('US/Eastern')

# calculate how much time to close
kalshi_w_markets['time_to_close'] = (kalshi_w_markets['close_time'] - kalshi_w_markets['timestamp']).dt.total_seconds()

# rank within each timestamp group so the smallest time_to_close gets rank 1 per timestamp
kalshi_w_markets['close_rank'] = kalshi_w_markets.groupby('timestamp')['time_to_close']\
    .rank(method='dense', ascending=True)

# take only closest ticker
kalshi_w_markets = kalshi_w_markets[kalshi_w_markets['close_rank'] == 1]


In [115]:
### Widen data to have it at timestamp level

# widen 
kalshi_df_wide = kalshi_w_markets.pivot_table(index=["timestamp"], columns="ticker_suffix", values=["volume_fp", "open_dollars", "high_dollars", "low_dollars", "close_dollars", "mean_dollars","open_interest_fp"]).reset_index()

# flatten multiindex columns into single level
kalshi_df_wide.columns = [
    "_".join(col).strip("_") if isinstance(col, tuple) else col
    for col in kalshi_df_wide.columns
]

# extract weekday and hour from timestamp
kalshi_df_wide['wd'] = kalshi_df_wide['timestamp'].dt.weekday 
kalshi_df_wide['hour'] = kalshi_df_wide['timestamp'].dt.hour

#categorize if markets are asleep
kalshi_df_wide['off_hours'] = (
((kalshi_df_wide['wd'] >= 0) & (kalshi_df_wide['wd'] <= 3) & (kalshi_df_wide['hour'] >= 17)) # Mon‑Thu after 5pm
| ((kalshi_df_wide['wd'] >= 1) & (kalshi_df_wide['wd'] <= 4) & (kalshi_df_wide['hour'] < 9)) # Tue‑Fri before 9am
| (kalshi_df_wide['wd'] >= 5) # Sat/Sun all day
| ((kalshi_df_wide['wd'] == 4) & (kalshi_df_wide['hour'] >= 17)) # Fri after 5pm (covers Fri‑Mon weekend block)
| ((kalshi_df_wide['wd'] == 0) & (kalshi_df_wide['hour'] < 9)) # Mon before 9am (end of weekend)
)

#categorize if markets have just closed
kalshi_df_wide['market_close'] = (
((kalshi_df_wide['wd'] >= 0) & (kalshi_df_wide['wd'] <= 4) & (kalshi_df_wide['hour'] == 17)) # Mon‑Thu after 5pm
# Mon before 9am (end of weekend)
)

#categorize if markets are bout to open
kalshi_df_wide['market_open'] = (
((kalshi_df_wide['wd'] >= 0) & (kalshi_df_wide['wd'] <= 4) & (kalshi_df_wide['hour'] == 9)) # Mon‑Thu at 9am # Mon before 9am (end of weekend)
)

#categorize if we are in weekend period
kalshi_df_wide['weekend_close_period'] = (
(kalshi_df_wide['wd'] >= 5) # Sat/Sun all day
| ((kalshi_df_wide['wd'] == 4) & (kalshi_df_wide['hour'] >= 17)) # Fri after 5pm (covers Fri‑Mon weekend block)
| ((kalshi_df_wide['wd'] == 0) & (kalshi_df_wide['hour'] < 9)) # Mon before 9am (end of weekend)
)

#get rid of unnecessary column describers
kalshi_df_wide.columns = (
    kalshi_df_wide.columns
    .str.replace('_fp_', '_', regex=False)
    .str.replace('_dollars_', '_', regex=False)
)


In [116]:
# fill in nas with 0 (we are assuming that na corresponds to a paused market hence 0 trading volume 0 price)
kalshi_df_wide = kalshi_df_wide.fillna(0)

# compute an expected rate column based on open interest (total volume across ticker lifetime) of trades per ticker (exclude extreme cuts/hikes)
kalshi_df_wide['exp_rate_open_interest'] = (
    -0.25 * kalshi_df_wide['open_interest_C25']
    + 0 * kalshi_df_wide['open_interest_H0']
    + 0.25 * kalshi_df_wide['open_interest_H25']
) / (
    kalshi_df_wide['open_interest_H0']
    + kalshi_df_wide['open_interest_H25']
    + kalshi_df_wide['open_interest_C25']
)

# compute an expected rate column based on volumes of trades per ticker (exclude extreme cuts/hikes)

kalshi_df_wide['exp_rate'] = (
    -0.25 * kalshi_df_wide['volume_C25']
    + 0 * kalshi_df_wide['volume_H0']
    + 0.25 * kalshi_df_wide['volume_H25']
) / (
    kalshi_df_wide['volume_H0']
    + kalshi_df_wide['volume_H25']
    + kalshi_df_wide['volume_C25']
)

kalshi_df_wide = kalshi_df_wide.fillna(0)

# !!!!!!!! Need to know how will aggregated sp data to merge
# Create a column with relevant date based on timestamp
# - weeknight after 5pm -> same day
# - weeknight before 9am -> previous day
# - weekend (Fri 5pm through Sun) -> following Monday
# - Monday before 9am also counts as Monday

def compute_relevant_date(ts):
    wd = ts.weekday()
    hour = ts.hour

    # weekend closed period assigns next Monday
    if wd == 4 and hour >= 17:  # Fri evening
        return (ts + pd.Timedelta(days=3)).date()
    if wd == 5:  # Saturday
        return (ts + pd.Timedelta(days=2)).date()
    if wd == 6:  # Sunday
        return (ts + pd.Timedelta(days=1)).date()

    # Monday before 9am still considered Monday
    #if wd == 0 and hour < 9:
    #    return ts.date()

    # normal weekday logic
    if hour >= 17:
        return (ts + pd.Timedelta(days=1)).date()
    if hour < 9:
        return ts.date()
    return ts.date()

kalshi_df_wide['relevant_date'] = kalshi_df_wide['timestamp'].apply(compute_relevant_date)

#kalshi_df_wide.to_csv("/Users/marcotaylhardat/Documents/ML/inf_check.csv", index=True)


In [117]:
### Filter to only keep off hours and not weekend close period (we want to see if we can predict overnight price changes based on volume and open interest patterns)
kalshi_df_wide_filtered = kalshi_df_wide[kalshi_df_wide['off_hours']]
kalshi_df_wide_filtered.to_csv("check.csv", index=True)
#kalshi_df_wide_filtered = kalshi_df_wide_filtered[~kalshi_df_wide_filtered['weekend_close_period']]

In [118]:
### Aggregate at overnight level

# Function to calculate % change from first non-zero value to last value in overnight period
def pct_change_start_end(series):
    if len(series) < 2:
        return 0
    
    # Find the first non-zero value
    non_zero_mask = series != 0
    if not non_zero_mask.any():
        return 0  # No non-zero values
    first_non_zero_idx = non_zero_mask.idxmax()  
    start_val = series.loc[first_non_zero_idx]
    end_val = series.iloc[-1]
    if start_val == 0:
        return 0  
    return (end_val - start_val) / start_val

# aggregate
kalshi_grouped_overnight = kalshi_df_wide_filtered.groupby('relevant_date').agg({
    'exp_rate': pct_change_start_end,
    'exp_rate_open_interest': pct_change_start_end,
    'volume_C25': 'sum',
    'volume_H0': 'sum',
    'volume_H25': 'sum',
    'mean_C25': pct_change_start_end,
    'mean_H0': pct_change_start_end,
    'mean_H25': pct_change_start_end,
})

print(kalshi_grouped_overnight.head())
print(kalshi_grouped_overnight['exp_rate_open_interest'].mean())

               exp_rate  exp_rate_open_interest  volume_C25  volume_H0  \
relevant_date                                                            
2025-07-31         -1.0                0.102547     20090.0    75482.0   
2025-08-01         -1.0               -1.000000     17523.0    36365.0   
2025-08-04         -1.0               -1.000000    137586.0   192290.0   
2025-08-05         -1.0                0.078286     70086.0    67412.0   
2025-08-06         -1.0                0.021213     27087.0    32074.0   

               volume_H25  mean_C25  mean_H0  mean_H25  
relevant_date                                           
2025-07-31           93.0      -1.0     -1.0      -1.0  
2025-08-01         2101.0      -1.0     -1.0      -1.0  
2025-08-04        11288.0      -1.0     -1.0      -1.0  
2025-08-05          108.0      -1.0     -1.0      -1.0  
2025-08-06          100.0      -1.0     -1.0      -1.0  
-0.024788073436184086


In [119]:
### Check that number of days seems right
print(kalshi_grouped_overnight.shape)
print(kalshi_grouped_overnight.head())


(162, 8)
               exp_rate  exp_rate_open_interest  volume_C25  volume_H0  \
relevant_date                                                            
2025-07-31         -1.0                0.102547     20090.0    75482.0   
2025-08-01         -1.0               -1.000000     17523.0    36365.0   
2025-08-04         -1.0               -1.000000    137586.0   192290.0   
2025-08-05         -1.0                0.078286     70086.0    67412.0   
2025-08-06         -1.0                0.021213     27087.0    32074.0   

               volume_H25  mean_C25  mean_H0  mean_H25  
relevant_date                                           
2025-07-31           93.0      -1.0     -1.0      -1.0  
2025-08-01         2101.0      -1.0     -1.0      -1.0  
2025-08-04        11288.0      -1.0     -1.0      -1.0  
2025-08-05          108.0      -1.0     -1.0      -1.0  
2025-08-06          100.0      -1.0     -1.0      -1.0  


In [120]:
kalshi_grouped_overnight.info()

<class 'pandas.core.frame.DataFrame'>
Index: 162 entries, 2025-07-31 to 2026-03-12
Data columns (total 8 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   exp_rate                162 non-null    float64
 1   exp_rate_open_interest  162 non-null    float64
 2   volume_C25              162 non-null    float64
 3   volume_H0               162 non-null    float64
 4   volume_H25              162 non-null    float64
 5   mean_C25                162 non-null    float64
 6   mean_H0                 162 non-null    float64
 7   mean_H25                162 non-null    float64
dtypes: float64(8)
memory usage: 11.4+ KB


In [121]:
kalshi_grouped_overnight.head()

,exp_rate,exp_rate_open_interest,volume_C25,volume_H0,volume_H25,mean_C25,mean_H0,mean_H25
relevant_date,,,,,,,,
2025-07-31,-1.0,0.102547,20090.0,75482.0,93.0,-1.0,-1.0,-1.0
2025-08-01,-1.0,-1.000000,17523.0,36365.0,2101.0,-1.0,-1.0,-1.0
2025-08-04,-1.0,-1.000000,137586.0,192290.0,11288.0,-1.0,-1.0,-1.0
2025-08-05,-1.0,0.078286,70086.0,67412.0,108.0,-1.0,-1.0,-1.0
2025-08-06,-1.0,0.021213,27087.0,32074.0,100.0,-1.0,-1.0,-1.0


In [122]:
macrot = pd.read_csv("../data/consolidated_data.csv", index_col = 0)

In [123]:
macrot.info()

<class 'pandas.core.frame.DataFrame'>
Index: 153 entries, 0 to 152
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Date                  153 non-null    object 
 1   Close                 153 non-null    float64
 2   High                  153 non-null    float64
 3   Low                   153 non-null    float64
 4   Open                  153 non-null    float64
 5   Volume                153 non-null    int64  
 6   Day Change %          153 non-null    float64
 7   Overnight Change %    153 non-null    float64
 8   3m_treasury           153 non-null    float64
 9   2yr_treasury          153 non-null    float64
 10  10yr_treasury         153 non-null    float64
 11  yield_curve           153 non-null    float64
 12  VIX                   153 non-null    float64
 13  fed_funds_rate        153 non-null    float64
 14  Overnight_Return      153 non-null    float64
 15  Overnight_Volatility  153 no

In [124]:
### Merge with macro data and save final dataset
#kalshi_grouped_overnight.to_csv("/Users/marcotaylhardat/Documents/ML/grouped_overnight_fix.csv", index=True)

# load in macro data
macro_df = pd.read_csv("../data/consolidated_data.csv", index_col = 0)
# convert Date to datetime then extract date to match kalshi format
macro_df['Date'] = pd.to_datetime(macro_df['Date']).dt.date
# reset index to make relevant_date a column, then rename
kalshi_grouped_overnight = kalshi_grouped_overnight.reset_index()
kalshi_grouped_overnight = kalshi_grouped_overnight.rename(columns={"relevant_date": "Date"})

# merge with macro data
kalshi_grouped_overnight = pd.merge(macro_df, kalshi_grouped_overnight, left_on='Date', right_on='Date', how='left')
print(kalshi_grouped_overnight.head())
    
kalshi_grouped_overnight.to_csv("../data/kalshi_w_macro_markets.csv", index=False)

         Date        Close         High          Low         Open      Volume  \
0  2025-07-31  6339.390137  6427.020020  6327.640137  6427.020020  6077080000   
1  2025-08-01  6238.009766  6287.279785  6212.689941  6287.279785  5827150000   
2  2025-08-04  6329.939941  6330.689941  6271.709961  6271.709961  4842580000   
3  2025-08-05  6299.189941  6346.000000  6289.370117  6336.629883  5517410000   
4  2025-08-06  6345.060059  6352.830078  6301.109863  6309.299805  5408560000   

   Day Change %  Overnight Change %  3m_treasury  2yr_treasury  ...  \
0     -1.363461            1.007718         4.41          3.94  ...   
1     -0.783646           -0.822009         4.35          3.69  ...   
2      0.928455            0.540240         4.35          3.69  ...   
3     -0.590849            0.105687         4.34          3.72  ...   
4      0.566786            0.160495         4.32          3.69  ...   

   Overnight_Volatility  Futures_Last_Price  exp_rate  exp_rate_open_interest  \
0    

In [125]:
kalshi_grouped_overnight.head()

,Date,Close,High,Low,Open,Volume,Day Change %,Overnight Change %,3m_treasury,2yr_treasury,...,Overnight_Volatility,Futures_Last_Price,exp_rate,exp_rate_open_interest,volume_C25,volume_H0,volume_H25,mean_C25,mean_H0,mean_H25
0,2025-07-31,6339.390137,6427.020020,6327.640137,6427.020020,6077080000,-1.363461,1.007718,4.41,3.94,...,0.002083,6456.75,-1.0,0.102547,20090.0,75482.0,93.0,-1.0,-1.0,-1.0
1,2025-08-01,6238.009766,6287.279785,6212.689941,6287.279785,5827150000,-0.783646,-0.822009,4.35,3.69,...,0.002764,6314.75,-1.0,-1.000000,17523.0,36365.0,2101.0,-1.0,-1.0,-1.0
2,2025-08-04,6329.939941,6330.689941,6271.709961,6271.709961,4842580000,0.928455,0.540240,4.35,3.69,...,0.000658,6304.00,-1.0,-1.000000,137586.0,192290.0,11288.0,-1.0,-1.0,-1.0
3,2025-08-05,6299.189941,6346.000000,6289.370117,6336.629883,5517410000,-0.590849,0.105687,4.34,3.72,...,0.001891,6365.75,-1.0,0.078286,70086.0,67412.0,108.0,-1.0,-1.0,-1.0
4,2025-08-06,6345.060059,6352.830078,6301.109863,6309.299805,5408560000,0.566786,0.160495,4.32,3.69,...,0.001797,6340.25,-1.0,0.021213,27087.0,32074.0,100.0,-1.0,-1.0,-1.0
